In [1]:
# [셀 1] Gemini 도구(google-genai) 설치

!pip install -q google-genai

In [2]:
# [셀 2] 열쇠함에서 API 키를 꺼내 Gemini 연결(client) 만들기

from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [ ]:
# [셀 3] (테스트) Gemini 첫 호출 - 연결 확인

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="안녕. 너를 간단히 소개해봐."
)
print(response.text)

안녕하세요, 저는 구글에서 개발한 인공지능 언어 모델입니다.


In [3]:
# [셀 4] 프롬프트 v2 - 카테고리 설명 추가, 긴급도 기준 명확화, 이름표 금지

def make_prompt(review):
    prompt = f"""너는 무인 스터디카페 사장이야.
아래 손님 리뷰를 읽고 4가지를 판단해줘.

[가게 정보]
- 가게 이름: 카공족 건대로데오점
- 요금: 24시간 이용권 5,900원(온라인 결제만 가능, 실물카드 불가)
- 무료 제공: 커피, 차, 아이스티, 얼음, 쿠키, 정수기
- 시설: 매장 내부 화장실(남녀공용 1개), 개인 스탠드, 독서대, 충전기
- 없는 것: 사물함, 스터디룸

[판단 기준]
1. 카테고리: 아래 7개 중 하나를 글자 그대로 고를 것 (목록에 없는 이름을 새로 만들지 말 것)
   - 청결: 청소 상태, 위생, 책상이나 화장실이 깨끗한지
   - 분위기·소음: 조용함, 집중 분위기, 음악, 다른 손님의 소음
   - 좌석·공간: 좌석 수, 자리 넓이, 혼잡함, 자리 잡기, 공간의 넓이와 채광
   - 냉난방: 실내 온도, 덥거나 추움, 에어컨과 난방
   - 시설·비품: 화장실 위치, 커피머신, 정수기, 와이파이, 충전기, 스탠드, 독서대, 사물함, 스터디룸
   - 결제·요금: 가격, 가성비, 결제 방식, 이용권 종류
   - 기타: 위 6개 어디에도 해당하지 않는 것
   - 여러 주제가 섞여 있으면 가장 중심이 되는 하나를 고를 것
   - 불만이 있는 리뷰는 불만의 대상이 되는 주제를 고를 것
2. 평가: 긍정 또는 부정
   - 불만이나 아쉬움이 조금이라도 섞여 있으면 부정
   - 시설이나 서비스가 있는지 묻거나 요청하는 질문도 부정
3. 긴급도: 높음, 보통, 낮음 중 하나
   - 높음: 고장이나 오류로 지금 매장 이용에 문제가 생긴 경우 (기계 고장, 와이파이 오류, 냉난방기 문제 등)
   - 보통: 운영 방식에 대한 불편이나 개선 요청 (결제 방식, 요금제, 혼잡, 시설 추가 요청)
   - 낮음: 칭찬이나 단순 의견
4. 답글: 사장 입장에서 쓴 3~4문장의 답글
   - 말투: 상냥하고 친절하되 과하지 않은 정중한 존댓말
   - 손님(유저)이름을 언급하지 말 것
   - 느낌표, 이모티콘을 문장 중간에 적절히 사용
   - 리뷰에서 손님이 언급한 내용을 구체적으로 짚어서 답할 것
   - 불만이 있는 리뷰에는 사과와 함께 확인하겠다는 내용을 넣을 것
   - 가게 정보에 없는 내용은 약속하지 말 것
   - 줄바꿈 없이 한 줄로 이어서 쓸 것

[출력 형식]
- "카테고리:", "평가:" 같은 이름표를 붙이지 말고 값만 쓸 것
- 다른 설명 없이 아래 형식의 한 줄만 출력할 것
카테고리|평가|긴급도|답글

[출력 예시]
시설·비품|부정|높음|이용에 불편을 드려 정말 죄송합니다😥 말씀해 주신 커피머신 고장은 바로 확인해서 조치하겠습니다. 쾌적하게 이용하실 수 있도록 더 꼼꼼히 관리할게요!

[리뷰]
{review}"""
    return prompt

In [ ]:
# [셀 5] (테스트) 리뷰 1개로 프롬프트 확인

review = "커피머신과 정수기가 작동하지 않습니다."

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=make_prompt(review)
)
print(response.text)

시설·비품|부정|높음|저희 매장의 커피머신과 정수기가 작동하지 않아 큰 불편을 드려 고개 숙여 사과드립니다.😭 해당 장비는 즉시 현장 점검을 실시하여 빠르게 정상화하도록 조치하겠습니다! 이용에 차질을 드린 점 다시 한번 죄송하며, 쾌적한 환경을 위해 더욱 노력하는 카공족 건대로데오점이 되겠습니다.🙏


In [ ]:
# [셀 6] (테스트) 결과를 | 기준으로 4칸 자르기

result = response.text.strip().split("|")

print(result)
print("카테고리:", result[0])
print("평가:", result[1])
print("긴급도:", result[2])
print("답글:", result[3])

['시설·비품', '부정', '높음', '저희 매장의 커피머신과 정수기가 작동하지 않아 큰 불편을 드려 고개 숙여 사과드립니다.😭 해당 장비는 즉시 현장 점검을 실시하여 빠르게 정상화하도록 조치하겠습니다! 이용에 차질을 드린 점 다시 한번 죄송하며, 쾌적한 환경을 위해 더욱 노력하는 카공족 건대로데오점이 되겠습니다.🙏']
카테고리: 시설·비품
평가: 부정
긴급도: 높음
답글: 저희 매장의 커피머신과 정수기가 작동하지 않아 큰 불편을 드려 고개 숙여 사과드립니다.😭 해당 장비는 즉시 현장 점검을 실시하여 빠르게 정상화하도록 조치하겠습니다! 이용에 차질을 드린 점 다시 한번 죄송하며, 쾌적한 환경을 위해 더욱 노력하는 카공족 건대로데오점이 되겠습니다.🙏


In [ ]:
# [셀 7] 리뷰 데이터(reviews.csv) 불러오기

import pandas as pd

df = pd.read_csv("reviews.csv", encoding="utf-8-sig")
print(len(df))
df.head()

90


,id,date,review,label,source
0,1,NaN,항상 잘 이용하고 있습니다. 앞으로도 자주 올게요,긍정,naver
1,2,NaN,화장실이 내부에 있어서 좋고 깔끔하게 관리가 잘 되고 있어요. 좋습니댱,긍정,naver
2,3,NaN,조용하고 깔끔해서 집중하기 좋아요,긍정,naver
3,4,NaN,매장이 깔끔하고 공부에 집중하기 너무 좋은 분위기였어요! 다음에 또 방문하고싶어요🤍,긍정,naver
4,5,NaN,생각했던 것보다 훨씬 좋아요!!!! 조용하고 모두 열심히 집중하고 계시니 저도 덩달...,긍정,naver


In [4]:
# [셀 8] Gemini에 묻는 함수 - 실패하면 30초 쉬고 최대 5번 재시도

import time

def ask_gemini(review):
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=make_prompt(review)
            )
            return response.text.strip()
        except Exception as e:
            print(attempt + 1, "번째 시도 실패:", e)
            time.sleep(30)
    return "오류|오류|오류|오류"

In [ ]:
# [셀 9] (테스트) 재시도 함수 확인 - 첫 번째 리뷰

print(ask_gemini(df["review"][0]))

기타|긍정|낮음|저희 카공족 건대로데오점을 늘 애용해 주시고 이렇게 따뜻한 리뷰까지 남겨주셔서 정말 감사합니다! 😊 항상 잘 이용하고 계신다는 말씀에 사장으로서 큰 힘과 보람을 느낍니다. ✨ 앞으로도 자주 방문해 주실 때마다 늘 쾌적하고 공부하기 좋은 공간이 될 수 있도록 정성껏 관리하겠습니다. ☕ 오늘도 목표하신 공부 모두 이루시고 행복한 하루 보내세요! 🙏


In [5]:
# [셀 10] 결과를 4칸으로 안전하게 자르는 함수 (4칸이 아니면 형식오류)

def split_result(text):
    parts = text.split("|")
    if len(parts) == 4:
        return [parts[0].strip(), parts[1].strip(), parts[2].strip(), parts[3].strip()]
    else:
        return ["형식오류", "형식오류", "형식오류", text]

In [ ]:
# [셀 11] (테스트) 리뷰 3개로 미리 확인

for review in df["review"].head(3):
    text = ask_gemini(review)
    parts = split_result(text)

    print("리뷰:", review)
    print("→", parts[0], "/", parts[1], "/", parts[2])
    print("→ 답글:", parts[3])
    print("----------")

    time.sleep(5)

리뷰: 항상 잘 이용하고 있습니다. 앞으로도 자주 올게요
→ 기타 / 긍정 / 낮음
→ 답글: 안녕하세요, 카공족 건대로데오점을 항상 애용해 주셔서 정말 감사합니다! 보내주신 따뜻한 말씀 덕분에 더욱 힘내서 운영하게 되네요😊 앞으로도 편안하고 쾌적한 공부 공간이 될 수 있도록 항상 노력할 테니, 언제든 필요할 때 편하게 찾아주세요!
----------
리뷰: 화장실이 내부에 있어서 좋고 깔끔하게 관리가 잘 되고 있어요. 좋습니댱
→ 시설·비품 / 긍정 / 낮음
→ 답글: 카공족 건대로데오점을 찾아주셔서 정말 감사합니다! 매장 내부 화장실이 깔끔하게 관리되고 있어 마음에 드셨다니 다행이에요. 앞으로도 쾌적하고 편안한 공간을 유지하기 위해 항상 노력하겠습니다! 공부하시다가 언제든 또 방문해 주세요!
----------
리뷰: 조용하고 깔끔해서 집중하기 좋아요
→ 좌석·공간 / 긍정 / 낮음
→ 답글: 카공족 건대로데오점에서 조용하고 깔끔한 분위기 속에서 집중이 잘 되셨다니 정말 기쁘네요! 앞으로도 편안하게 공부하실 수 있도록 항상 쾌적한 환경을 유지하기 위해 노력하겠습니다. 다음에도 열공하러 또 찾아주세요~!
----------


In [ ]:
# [셀 12] 리뷰 90개 전체 분석

categories = []
ai_labels = []
urgencies = []
replies = []

count = 0
for review in df["review"]:
    text = ask_gemini(review)
    parts = split_result(text)

    categories.append(parts[0])
    ai_labels.append(parts[1])
    urgencies.append(parts[2])
    replies.append(parts[3])

    count = count + 1
    print(count, "/", len(df), "완료:", parts[0], parts[1], parts[2])

    time.sleep(5)

1 / 90 완료: 기타 긍정 낮음
2 / 90 완료: 시설·비품 긍정 낮음
3 / 90 완료: 분위기·소음 긍정 낮음
4 / 90 완료: 기타 긍정 낮음
5 / 90 완료: 분위기·소음 긍정 낮음
6 / 90 완료: 분위기·소음 긍정 낮음
7 / 90 완료: 기타 긍정 낮음
8 / 90 완료: 분위기·소음 긍정 낮음
9 / 90 완료: 결제·요금 긍정 낮음
10 / 90 완료: 기타 긍정 낮음
11 / 90 완료: 좌석·공간 부정 보통
12 / 90 완료: 기타 긍정 낮음
13 / 90 완료: 시설·비품 긍정 낮음
14 / 90 완료: 좌석·공간 긍정 낮음
15 / 90 완료: 분위기·소음 긍정 낮음
16 / 90 완료: 기타 긍정 낮음
17 / 90 완료: 기타 긍정 낮음
18 / 90 완료: 결제·요금 긍정 낮음
19 / 90 완료: 기존·비품 긍정 낮음
20 / 90 완료: 분위기·소음 긍정 낮음
21 / 90 완료: 결제·요금 긍정 낮음
22 / 90 완료: 기타 긍정 낮음
23 / 90 완료: 기타 긍정 낮음
24 / 90 완료: 분위기·소음 긍정 낮음
25 / 90 완료: 좌석·공간 부정 보통
26 / 90 완료: 분위기·소음 긍정 낮음
27 / 90 완료: 분위기·소음 긍정 낮음
28 / 90 완료: 기타 긍정 낮음
29 / 90 완료: 기타 긍정 낮음
30 / 90 완료: 결제·요금 긍정 낮음
31 / 90 완료: 결제·요금 긍정 낮음
32 / 90 완료: 좌석·공간 부정 보통
33 / 90 완료: 분위기·소음 긍정 낮음
34 / 90 완료: 기타 긍정 낮음
35 / 90 완료: 분위기·소음 긍정 낮음
36 / 90 완료: 결제·요금 긍정 낮음
37 / 90 완료: 결제·요금 부정 보통
38 / 90 완료: 시설·비품 부정 보통
39 / 90 완료: 좌석·공간 긍정 낮음
40 / 90 완료: 시설·비품 긍정 낮음
41 / 90 완료: 분위기·소음 긍정 낮음
42 / 90 완료: 청결 긍정 낮음
43 / 90 완료: 시설·비품 긍정 낮음
44 /

In [ ]:
# [셀 13] 분석 결과를 표에 붙이고 results.csv로 저장

df["category"] = categories
df["ai_label"] = ai_labels
df["urgency"] = urgencies
df["reply"] = replies

df.to_csv("results_v2.csv", index=False, encoding="utf-8-sig")

print(df["ai_label"].value_counts())
df.head()

ai_label
긍정    76
부정    14
Name: count, dtype: int64


,id,date,review,label,source,category,ai_label,urgency,reply
0,1,NaN,항상 잘 이용하고 있습니다. 앞으로도 자주 올게요,긍정,naver,기타,긍정,낮음,항상 카공족 건대로데오점을 찾아주셔서 정말 감사합니다😊 앞으로도 편안하고 집중 잘 ...
1,2,NaN,화장실이 내부에 있어서 좋고 깔끔하게 관리가 잘 되고 있어요. 좋습니댱,긍정,naver,시설·비품,긍정,낮음,매장 내부 화장실을 편리하게 이용해 주시고 깔끔하다고 칭찬해 주셔서 정말 감사합니다...
2,3,NaN,조용하고 깔끔해서 집중하기 좋아요,긍정,naver,분위기·소음,긍정,낮음,조용하고 깔끔한 분위기 속에서 집중이 잘 되셨다니 정말 기쁩니다! 앞으로도 언제나 ...
3,4,NaN,매장이 깔끔하고 공부에 집중하기 너무 좋은 분위기였어요! 다음에 또 방문하고싶어요🤍,긍정,naver,기타,긍정,낮음,매장이 깔끔하고 공부하기 좋은 분위기였다니 정말 기쁩니다😊 마음에 드셨다니 다행이에...
4,5,NaN,생각했던 것보다 훨씬 좋아요!!!! 조용하고 모두 열심히 집중하고 계시니 저도 덩달...,긍정,naver,분위기·소음,긍정,낮음,조용하고 집중하는 매장 분위기를 마음에 들어 해 주셔서 정말 기쁩니다! 준비해 드린...


In [ ]:
# [셀 14] (확인) 결과값 개수 세기 - 이상한 값 찾기

print(df["category"].value_counts())
print(df["urgency"].value_counts())
df[df["ai_label"] == "감정: 긍정"]

category
분위기·소음    20
시설·비품     19
기타        16
좌석·공간     16
결제·요금     14
청결         3
기존·비품      1
냉난방        1
Name: count, dtype: int64
urgency
낮음    76
보통    11
높음     3
Name: count, dtype: int64


,id,date,review,label,source,category,ai_label,urgency,reply


In [ ]:
# [셀 15] 전처리 - 이름표와 잘못된 카테고리 정리 후 다시 저장 (v2 → results_v2.csv)

# ① 이름표 지우기 (혹시 남아 있을 경우를 대비)
df["category"] = df["category"].str.replace("카테고리:", "").str.strip()
df["ai_label"] = df["ai_label"].str.replace("감정:", "").str.replace("평가:", "").str.strip()
df["urgency"] = df["urgency"].str.replace("긴급도:", "").str.strip()
df["reply"] = df["reply"].str.replace("답글:", "").str.strip()

# ② 목록에 없는 카테고리 고치기
df["category"] = df["category"].str.replace("기분·소음", "분위기·소음")
df["category"] = df["category"].str.replace("시설·공간", "시설·비품")
df["category"] = df["category"].str.replace("기존·비품", "시설·비품")

# ③ 다시 저장하고 확인
df.to_csv("results_v2.csv", index=False, encoding="utf-8-sig")
print(df["category"].value_counts())
print(df["ai_label"].value_counts())
print(df["urgency"].value_counts())

category
시설·비품     20
분위기·소음    20
기타        16
좌석·공간     16
결제·요금     14
청결         3
냉난방        1
Name: count, dtype: int64
ai_label
긍정    76
부정    14
Name: count, dtype: int64
urgency
낮음    76
보통    11
높음     3
Name: count, dtype: int64


In [ ]:
# [셀 16] v1에서 문제였던 리뷰 6개의 v2 결과 확인

check_ids = [12, 16, 38, 55, 83, 84]
df[df["id"].isin(check_ids)][["id", "review", "category", "ai_label", "urgency"]]

,id,review,category,ai_label,urgency
11,12,조용하고 깨끗한 분위가 조성되어있어 이용에 편리합니다.! 자격증준비하느라 집중할만한...,기타,긍정,낮음
15,16,너무 깔끔하고 좋습니다! 집 주변에 있어서 좋네요! 가격도 저렴하고 딱입니다 ㅠㅠ,기타,긍정,낮음
37,38,잔잔한 노래 나와서 노트북이나 계산기 편하게 치면서 공부할 수 있어요 공부하시는 분...,시설·비품,부정,보통
54,55,시설이 깔끔하고 조용해서 공부하기 정말 좋았어요! 전체적으로 쾌적해서 오래 있어도 ...,분위기·소음,긍정,낮음
82,83,다른곳은 시간권도 있던데 여기도 생기면 좋겠네요,결제·요금,부정,보통
83,84,실물 카드 결제가 안되는게 너무 불편해요,결제·요금,부정,보통


In [ ]:
# [셀 17] v2에서 '부정'으로 분류된 리뷰의 카테고리·긴급도 확인

df[df["ai_label"] == "부정"][["id", "review", "label", "category", "urgency"]]

,id,review,label,category,urgency
10,11,매주 토요일마다 올 예정이에요~~!! 사람이 많아져서 아쉽습니다 ㅠㅠㅠ,부정,좌석·공간,보통
24,25,24시간 이용에 5900원이라서 가성비좋은거 같아요 다만 사람이 좀 많아서 자리 잡...,부정,좌석·공간,보통
31,32,"생각보다 사람이 많아서 조금 놀랐고, 공간이 조금 부족하다는 느낌이 들면서 답답함이...",부정,좌석·공간,보통
36,37,맨날 어린이대공원역으로만 갔는데 더 가까운 곳이 생겨서 너무 좋아요!! 이제 성수역...,부정,결제·요금,보통
37,38,잔잔한 노래 나와서 노트북이나 계산기 편하게 치면서 공부할 수 있어요 공부하시는 분...,부정,시설·비품,보통
62,63,스터디카페가 조금 답답하고 카페는 시끄러울때 가기 딱 좋습니다. 좌석도 충분히 많아...,부정,시설·비품,보통
82,83,다른곳은 시간권도 있던데 여기도 생기면 좋겠네요,부정,결제·요금,보통
83,84,실물 카드 결제가 안되는게 너무 불편해요,부정,결제·요금,보통
84,85,너무 시끄럽게 과자를 드시는 분이 계셔서 다른 이용자들도 쳐다볼 정도로 소음이 심했어요,부정,분위기·소음,보통
85,86,너무 추워요,부정,냉난방,높음


In [6]:
# [셀 18] Gradio(화면 만드는 도구) 설치

!pip install -q gradio

In [10]:
# [셀 19] UI용 함수 - 결과 정리, 리뷰 1개 분석, 파일 분석

import time
import gradio as gr

# AI 결과에서 이름표를 지우고 잘못된 카테고리를 고치는 함수 (셀 15 전처리와 같은 역할)
def clean_parts(parts):
    category, label, urgency, reply = parts
    category = category.replace("카테고리:", "").strip()
    label = label.replace("감정:", "").replace("평가:", "").strip()
    urgency = urgency.replace("긴급도:", "").strip()
    reply = reply.replace("답글:", "").strip()
    typo = {"기분·소음": "분위기·소음", "시설·공간": "시설·비품", "기존·비품": "시설·비품"}
    category = typo.get(category, category)
    return category, label, urgency, reply

# 화면에 보여줄 아이콘
LABEL_ICON = {"긍정": "🟢 긍정", "부정": "🔴 부정"}
URGENCY_ICON = {"높음": "🚨 높음", "보통": "⚠️ 보통", "낮음": "✅ 낮음"}

# 불만 카테고리 가로 막대그래프를 HTML로 그리는 함수
def make_chart_html(chart):
    if len(chart) == 0:
        return "<p>불만 리뷰가 없어요 🎉</p>"
    max_count = chart["불만 수"].max()
    html = "<div style='padding:8px 4px'>"
    html += "<div style='font-weight:600;margin-bottom:12px'>불만 카테고리</div>"
    for _, row in chart.iterrows():
        name = row["카테고리"]
        count = row["불만 수"]
        width = count / max_count * 100
        html += "<div style='display:flex;align-items:center;margin:8px 0'>"
        html += f"<div style='width:100px;font-size:14px'>{name}</div>"
        html += "<div style='flex:1;background:rgba(128,128,128,0.15);border-radius:6px;height:20px'>"
        html += f"<div style='width:{width}%;background:#e8743b;height:100%;border-radius:6px'></div>"
        html += "</div>"
        html += f"<div style='width:50px;text-align:right;font-weight:600'>{count}개</div>"
        html += "</div>"
    html += "</div>"
    return html

# 탭 1: 리뷰 하나 분석
def analyze_one(review):
    if review is None or review.strip() == "":
        return "-", "-", "-", "리뷰를 입력해 주세요."
    parts = split_result(ask_gemini(review))
    category, label, urgency, reply = clean_parts(parts)
    return category, LABEL_ICON.get(label, label), URGENCY_ICON.get(urgency, urgency), reply

# 탭 2: CSV 파일 전체 분석 (요약 + 불만 그래프 + 결과 표 + 다운로드)
def analyze_file(file, progress=gr.Progress()):
    if file is None:
        return "파일을 올려 주세요.", "", None, None
    path = file if isinstance(file, str) else file.name

    # UTF-8로 먼저 읽어보고, 안 되면 윈도우 한글 방식(cp949)으로 다시 읽기
    try:
        data = pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        data = pd.read_csv(path, encoding="cp949")

    if "review" not in data.columns:
        return "`review` 칸이 있는 CSV 파일을 올려 주세요.", "", None, None

    reviews = data["review"].fillna("").astype(str).tolist()
    rows = []
    for i, review in enumerate(reviews):
        progress((i + 1) / len(reviews), desc=f"{i + 1} / {len(reviews)} 분석 중")
        if review.strip() == "":
            continue
        category, label, urgency, reply = clean_parts(split_result(ask_gemini(review)))
        rows.append({"긴급도": urgency, "평가": label, "카테고리": category, "리뷰": review, "답글": reply})
        time.sleep(5)

    result = pd.DataFrame(rows)
    order = {"높음": 0, "보통": 1, "낮음": 2}
    result["순서"] = result["긴급도"].map(order).fillna(3)
    result = result.sort_values("순서", kind="stable").drop(columns="순서")

    total = len(result)
    negative = (result["평가"] == "부정").sum()
    urgent = (result["긴급도"] == "높음").sum()
    summary = f"### 📊 분석 요약\n전체 **{total}개** · 불만 **{negative}개** · 긴급 **{urgent}개**"

    # 불만 리뷰만 골라서 카테고리별 개수 세기 (KNIME의 Row Filter → Bar Chart와 같은 일)
    chart = result[result["평가"] == "부정"]["카테고리"].value_counts().reset_index()
    chart.columns = ["카테고리", "불만 수"]
    chart_html = make_chart_html(chart)

    out_path = "review_bot_results.csv"
    result.to_csv(out_path, index=False, encoding="utf-8-sig")
    return summary, chart_html, result, out_path

In [11]:
# [셀 20] Gradio 화면 구성 및 실행 (실행하면 공유 링크가 생김)

examples = [
    "조용하고 깨끗해서 공부하기 정말 좋아요!",
    "분위기는 좋은데 사람이 너무 많아서 자리 잡기가 힘들어요",
    "와이파이가 계속 끊겨서 인강을 못 들었어요",
]

with gr.Blocks() as demo:
    gr.Markdown("# 리뷰 관리 도우미\n리뷰를 넣으면 AI가 **카테고리 · 평가 · 긴급도**를 분류하고 **답글 초안**을 작성해요.")

    with gr.Tab("📝 리뷰 하나 분석"):
        review_in = gr.Textbox(label="리뷰", lines=4, placeholder="손님 리뷰를 붙여넣으세요")
        gr.Examples(examples=examples, inputs=review_in, label="예시 리뷰")
        one_btn = gr.Button("🔍 분석하기", variant="primary")
        with gr.Row():
            category_out = gr.Textbox(label="카테고리", interactive=False)
            label_out = gr.Textbox(label="평가", interactive=False)
            urgency_out = gr.Textbox(label="긴급도", interactive=False)
        reply_out = gr.Textbox(label="답글 초안", lines=4, interactive=False, buttons=["copy"])

        # ⭐ 버튼을 누르면 → analyze_one() 실행 → 결과 4개를 화면 4칸에 표시
        one_btn.click(fn=analyze_one, inputs=review_in,
                      outputs=[category_out, label_out, urgency_out, reply_out])

    with gr.Tab("📂 파일로 한꺼번에 분석"):
        gr.Markdown("`review` 칸이 있는 CSV 파일을 올리면 모든 리뷰를 분석하고, **긴급한 리뷰부터** 정렬해서 보여줘요.")
        file_in = gr.File(label="리뷰 CSV 파일", file_types=[".csv"])
        file_btn = gr.Button("🔍 전체 분석", variant="primary")
        summary_out = gr.Markdown()
        chart_out = gr.HTML()
        table_out = gr.Dataframe(label="분석 결과")
        download_out = gr.File(label="결과 파일 다운로드")

        # ⭐ 버튼을 누르면 → analyze_file() 실행 → 요약, 불만 그래프, 표, 다운로드 파일 표시
        file_btn.click(fn=analyze_file, inputs=file_in,
                       outputs=[summary_out, chart_out, table_out, download_out])

    gr.Markdown("<sub>Gemini 3.5 Flash-Lite · 프롬프트 v2 · KNIME 검증 결과(리뷰 90개): 긍정/부정 정확도 100%, 불만 리뷰 재현율 100%</sub>")

demo.launch(share=True, show_error=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11ee3afe9a32324aa8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
